# 03. Recursive Closure And Explain

This notebook introduces the part of AETHER that makes it more than a ledger: recursive derivation.

We will model a dependency chain, derive the transitive closure, and then ask the kernel to explain one of the derived tuples.


In [ ]:
from pathlib import Path
import subprocess
import sys

candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
repo_root = next(
    (
        path
        for path in candidate_roots
        if (path / "python").exists() and (path / "Cargo.toml").exists()
    ),
    None,
)

if repo_root is None:
    repo_root = Path("/content/AETHER")
    if not repo_root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/fyremael/AETHER", str(repo_root)],
            check=True,
        )

python_root = repo_root / "python"
if str(python_root) not in sys.path:
    sys.path.insert(0, str(python_root))

repo_root


In [ ]:
from notebooks.colab_setup import ensure_rust_toolchain, pretty_json, start_http_service, stop_http_service
from aether_sdk import AetherClient, make_datom, value_entity

ensure_rust_toolchain()
session = start_http_service(repo_root)
client = AetherClient(session.base_url)

pretty_json(client.health())


## Append The Extensional Dependency Facts

Each datom says that one task depends on another.


In [ ]:
client.append(
    [
        make_datom(entity=1, attribute=1, value=value_entity(2), element=1, op="Add"),
        make_datom(entity=2, attribute=1, value=value_entity(3), element=2, op="Add"),
        make_datom(entity=3, attribute=1, value=value_entity(4), element=3, op="Add"),
    ]
)

pretty_json(client.history())


In [ ]:
recursive_document = """
schema v1 {
  attr task.depends_on: RefSet<Entity>
}

predicates {
  task_depends_on(Entity, Entity)
  depends_transitive(Entity, Entity)
}

rules {
  depends_transitive(x, y) <- task_depends_on(x, y)
  depends_transitive(x, z) <- depends_transitive(x, y), task_depends_on(y, z)
}

materialize {
  depends_transitive
}

query reachability {
  current
  goal depends_transitive(x, y)
  keep x, y
}
"""

reachability = client.run_named_query(recursive_document, query_name="reachability")
pretty_json(reachability)


In [ ]:
target_row = next(
    row
    for row in reachability["rows"]
    if row["values"] == [{"Entity": 1}, {"Entity": 4}]
)
tuple_id = target_row["tuple_id"]
trace = client.explain_tuple(tuple_id)

print(f"Explaining tuple_id={tuple_id}")
pretty_json(trace)


## What The Trace Means

The explanation ties the derived tuple back to the source datoms and intermediate tuples that made it true.

That is why AETHER can do more than compute an answer: it can defend the answer.


In [ ]:
# Uncomment this cell when you are done with the notebook.
# stop_http_service(session)
